<a href="https://colab.research.google.com/drive/1U0XcubWNl7397PYx3cFpGxd7NIg2Kioz?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![CuratorKIT](CuratorKIT_logo-TP.png)

---

# Synthetic generation — GRPO

**Exporter:** `grpo` &nbsp;|&nbsp; **Gates:** HallucinationGate + RewardGate (Stage 1) &nbsp;|&nbsp; **Scoring:** LLM reward (Stage 2)

GRPO needs prompts with multiple candidate responses, each scored. This requires
two `Curator.run()` calls:

1. **Stage 1:** PDF → QA questions (filtered by HallucinationGate + RewardGate)
2. **Stage 2:** Questions → N rollout responses at varied temperatures + reward scores

### Why two stages?
Raw PDF chunks aren't prompts. Generating rollouts directly from a chunk produces
N summaries, not N candidate answers to a specific question. Stage 1 converts
chunks into question prompts; Stage 2 generates the rollouts.

### Valid task for GRPO export

| Stage 1 task | Stage 2 task | Output |
|-------------|-------------|--------|
| `qa` | `grpo` | N responses per question + reward scores |

### Pipeline
```
Stage 1: PDF → chunk → dedup → clean → qa → HallGate → RewardGate → alpaca.jsonl
Stage 2: alpaca.jsonl → grpo (N rollouts at spread temps) → reward scoring → grpo.jsonl
```

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")

# !pip install "/content/curatorkit.tar.gz[generation,embedding,hf,parquet,pdf-gpu]" nest-asyncio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 1. Configuration

In [2]:
# Demo PDF
import urllib.request
urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "attention_is_all_you_need.pdf")
print("Downloaded.")

Downloaded.


## LLM Backend Setup

Configure your LLM endpoint before running. CuratorKIT uses [LiteLLM](https://docs.litellm.ai) under the hood, so any OpenAI-compatible endpoint works.

### Option A — vLLM (recommended for speed)

```bash
# Serve a local model with vLLM
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000
```

Then set in the config cell below:
```python
MODEL    = "openai/Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
```

### Option B — Ollama (easiest local setup)

```bash
# Install Ollama: https://ollama.com
ollama pull qwen2.5:7b
ollama serve  # starts on port 11434 by default
```

Then set:
```python
MODEL    = "ollama/qwen2.5:7b"
API_BASE = ""  # Ollama backend is auto-detected from the model prefix
```

### Option C — Any OpenAI-compatible API

```python
MODEL    = "openai/your-model-name"
API_BASE = "https://your-api-endpoint/v1"
# Set API key via env var: export OPENAI_API_KEY=your-key
# Or pass directly: llm_api_key="your-key" in CuratorConfig
```


In [3]:
MODEL    = "openai/Qwen/Qwen3-8B"
JUDGE    = "openai/Qwen/Qwen3-8B"
API_BASE = " "

In [9]:
from pathlib import Path
from curatorkit import Curator, CuratorConfig

# ── PDF source ────────────────────────────────────────────────────────────
PDF_PATH = "attention_is_all_you_need.pdf"  # ← or replace with your own PDF


# ── Pipeline params ───────────────────────────────────────────────────────
MAX_CHUNKS    = 10
NUM_QUESTIONS = 3      # questions per chunk (stage 1)
NUM_RESPONSES = 4      # rollout candidates per question (stage 2)
TEMP_SPREAD   = 0.6    # temperature range across rollouts
CONCURRENCY   = 32
OUTPUT_DIR    = Path("output/grpo")
STAGE1_DIR    = OUTPUT_DIR / "stage1"

print(f"PDF        : {PDF_PATH}")
print(f"Model      : {MODEL}")
print(f"Questions  : {NUM_QUESTIONS}/chunk  →  {NUM_RESPONSES} rollouts each")
print(f"Temp spread: {TEMP_SPREAD}")
print(f"Stage 1    : {STAGE1_DIR}/")
print(f"Final      : {OUTPUT_DIR}/")

PDF        : attention_is_all_you_need.pdf
Model      : openai/Qwen/Qwen3-8B
Questions  : 3/chunk  →  4 rollouts each
Temp spread: 0.6
Stage 1    : output/grpo/stage1/
Final      : output/grpo/


## 2. Stage 1: PDF → QA Questions (both gates filter prompts)

In [10]:
api_base = API_BASE if API_BASE else None

stage1_config = CuratorConfig(
    dataset={"name": PDF_PATH, "max_samples": MAX_CHUNKS},
    llm_api_key=" ",
    pdf_chunk_strategy="heading",
    pdf_chunk_max_tokens=512,
    pdf_chunk_overlap_tokens=50,
    dedup="minhash",
    minhash_threshold=0.85,
    clean=True,
    llm_model=MODEL,
    llm_api_base=api_base,
    llm_max_tokens=2048,
    llm_concurrency=CONCURRENCY,
    generation_concurrency=CONCURRENCY,
    llm_extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    judge_llm_model=JUDGE,
    judge_llm_api_base=api_base,
    judge_llm_temperature=0.1,
    judge_concurrency=CONCURRENCY,
    judge_llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    generation_task="qa",
    num_questions=NUM_QUESTIONS,
    difficulty="medium",
    hallucination_threshold=0.5, # Can change
    reward_threshold=0.6, # Can change as per your needs
    reward_dimensions=["helpfulness", "honesty", "instruction_following"],
    export_formats=["alpaca"],
    output_dir=STAGE1_DIR,
)

print("Stage 1: PDF → QA questions (HallucinationGate + RewardGate)...")
r1 = Curator(stage1_config).run()
r1.print_summary()

qa_file = STAGE1_DIR / "sft_alpaca.jsonl"
print(f"\nQuestions saved : {qa_file}")
print(f"Passed gates    : {len(r1.passed):,} questions")

Stage 1: PDF → QA questions (HallucinationGate + RewardGate)...


2026-06-12 06:01:38.822 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:209 - Pipeline processing-window multi-file run. doc_count=1, total_pages=15, window_size=64, total_batches=1
2026-06-12 06:01:38.822 | DEBUG    | mineru.utils.pdf_image_tools:_load_images_from_pdf_bytes_range:289 - PDF image rendering uses 1 processes for pages 1-15: [(0, 14)]
2026-06-12 06:01:40.465 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:264 - Pipeline processing window batch 1/1: 15/15 pages, batch_pages=15, doc_slices=doc0:1-15
2026-06-12 06:01:40.466 | INFO     | mineru.backend.pipeline.pipeline_analyze:batch_image_analyze:364 - GPU Memory: 15 GB, Batch Ratio: 4. 
OCR-det en: 100%|██████████| 30/30 [00:02<00:00, 12.83it/s]
Seal Predict: 0it [00:00, ?it/s]
Processing pages: 100%|██████████| 15/15 [00:01<00:00,  9.01it/s]
2026-06-12 06:01:46.693 | DEBUG    | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:320 - processing-window mult

────────────────────────────────────────────
  passed   :        2
  rejected :       54
  time     :    66.1s
  output   : output/grpo/stage1
────────────────────────────────────────────

Questions saved : output/grpo/stage1/sft_alpaca.jsonl
Passed gates    : 2 questions


## 3. Stage 2: Questions → Rollouts + Reward Scores

In [11]:
if not qa_file.exists() or not r1.passed:
    print("ERROR: Stage 1 produced no output. Check PDF path and LLM endpoint.")
else:
    print(f"Reading {len(r1.passed):,} questions from {qa_file}")

    # Stage 2 reads the generated questions file (not a PDF).
    # The alpaca JSONL is auto-detected; GRPORolloutTask uses instruction as prompt.
    stage2_config = CuratorConfig(
        dataset=str(qa_file),
        llm_model=MODEL,
        llm_api_base=api_base,
        llm_api_key=" ",
        llm_max_tokens=2048,
        llm_concurrency=CONCURRENCY,
        generation_concurrency=CONCURRENCY,
        llm_extra_body={"chat_template_kwargs": {"enable_thinking": True}},
        generation_task="grpo",
        num_responses=NUM_RESPONSES,
        score_responses=True,
        grpo_temperature_spread=TEMP_SPREAD,
        grpo_scoring_llm_model=JUDGE,
        export_formats=["grpo"],
        output_dir=OUTPUT_DIR,
    )

    print("\nStage 2: Questions → N rollouts + reward scores...")
    r2 = Curator(stage2_config).run()
    r2.print_summary()

Reading 2 questions from output/grpo/stage1/sft_alpaca.jsonl

Stage 2: Questions → N rollouts + reward scores...


[GRPORolloutTask] generating: 100%|██████████| 2/2 [00:13<00:00,  6.55s/sample]

────────────────────────────────────────────
  passed   :        0
  rejected :        2
  time     :    13.1s
  output   : output/grpo
────────────────────────────────────────────


## 4. Inspect

In [13]:
import statistics

if 'r2' in dir() and r2.passed:
    print(f"GRPO groups: {len(r2.passed):,}\n")

    for i, s in enumerate(r2.passed[:2]):
        print(f"── Group {i+1} ──")
        print(f"  prompt: {s.instruction[:120]}")
        for j, (resp, score) in enumerate(zip(s.responses, s.reward_scores)):
            print(f"  [{j}] score={score:.2f}  {resp[:80]}")
        if s.reward_scores:
            best = max(s.reward_scores)
            worst = min(s.reward_scores)
            print(f"  spread: {best - worst:.2f}  (best={best:.2f}  worst={worst:.2f})")
        print()

    # ── Score distribution ─────────────────────────────────────────────────
    all_scores = [s for sample in r2.passed for s in sample.reward_scores]
    print(f"Score distribution ({len(all_scores)} rollouts):")
    print(f"  mean={statistics.mean(all_scores):.3f}  median={statistics.median(all_scores):.3f}")
    print(f"  min={min(all_scores):.3f}  max={max(all_scores):.3f}  stdev={statistics.stdev(all_scores):.3f}")

## 5. How GRPO differs from Alpaca/DPO

| Aspect | Alpaca | DPO | GRPO |
|--------|--------|-----|------|
| **Gates** | Hallucination + Reward | Hallucination + Reward (dual-score) | Hallucination + Reward (Stage 1 only) |
| **Stage 2 filtering** | N/A | N/A | Reward scoring — ranking, not filtering |
| **All samples kept?** | No — gates reject | No — gates reject | Yes — all rollouts kept, scores provide ranking |

GRPO training uses `(best_response - mean) / std` as advantage — the **score spread**
matters more than absolute scores. A group with scores [0.9, 0.5, 0.4, 0.3] gives
stronger training signal than [0.7, 0.68, 0.65, 0.63].

### Variations
- **More rollouts:** `num_responses=8`
- **Wider spread:** `grpo_temperature_spread=1.0`
- **Explicit temps:** `grpo_temperatures=[0.3, 0.7, 1.0, 1.3]`
- **Separate scoring model:** `grpo_scoring_llm_model="openai/gpt-4o-mini"`